# Fantasy Football Scout Data Scraper

This notebook automates the process of scraping player statistics from Fantasy Football Scout's custom stats tables for the 2024/25 season. The data is saved as a CSV file for further analysis.

## 1. Import Required Libraries

Import all necessary libraries, including time, pandas, pathlib, and Selenium modules.

In [87]:
import time
import pandas as pd
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.firefox import GeckoDriverManager
from selenium.common.exceptions import TimeoutException

## 2. Define Constants and Output Path

Define constants such as `USERNAME`, `PASSWORD`, `SEASON`, and `OUTPUT_PATH`. Create the output directory if it doesn't exist.

In [88]:
# --- Ρυθμίσεις ---
USERNAME = "ritaras4"
PASSWORD = "FSUfpldata!"
SEASON = "2024/25"
OUTPUT_PATH = Path("framework/data/fpl_scout/2024-25")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

## 3. Initialize Selenium WebDriver

Set up the Selenium WebDriver using GeckoDriverManager and initialize WebDriverWait.

In [89]:
# --- Ξεκινάμε Selenium ---
driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()))
wait = WebDriverWait(driver, 20)

In [90]:
# -----------------------------
# 1. ΠΛΟΗΓΗΣΗ & ΑΠΟΔΟΧΗ COOKIES
# -----------------------------
driver.get("https://www.fantasyfootballscout.co.uk/")

try:
    # Κάποια sites αλλάζουν όνομα κουμπιού, οπότε δοκιμάζουμε 2 περιπτώσεις
    cookie_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(text(), 'Accept') or contains(text(), 'accept')]")
        )
    )
    cookie_btn.click()
    print("✅ Cookies accepted.")
except TimeoutException:
    print("⚠️ No cookie button appeared (maybe already accepted).")

⚠️ No cookie button appeared (maybe already accepted).


## 4. Login to Fantasy Football Scout

Navigate to the login page, input credentials, and log in using Selenium.

In [91]:
# -----------------------------
# SIMPLE & ROBUST LOGIN (updated Nov 2025)
# -----------------------------
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

driver.get("https://members.fantasyfootballscout.co.uk/")
wait = WebDriverWait(driver, 15)

USERNAME = "ritaras4"
PASSWORD = "FSUfpldata!"

# 1. Accept cookies if visible
try:
    cookie = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(.,'Accept') or contains(.,'ACCEPT')]"))
    )
    cookie.click()
    print("✅ Cookies accepted")
    time.sleep(1)
except Exception:
    print("ℹ️ No cookies to accept")

# 2. Click "Log in" button on homepage
try:
    login_btn = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//a[contains(.,'Log in') or contains(.,'Sign in')]"))
    )
    ActionChains(driver).move_to_element(login_btn).click().perform()
    print("➡️ Clicked 'Log in' button")
except Exception as e:
    print("⚠️ Couldn't find login button:", e)

# 3. Wait for the login form to appear
try:
    username_input = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//input[@type='text' or @name='log' or contains(@id,'user_login')]"))
    )
    password_input = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//input[@type='password' or @name='pwd' or contains(@id,'user_pass')]"))
    )
    print("✅ Login form detected")
except Exception as e:
    raise RuntimeError("⚠️ Login form not found — site layout may have changed.") from e

# 4. Fill in credentials
username_input.clear()
username_input.send_keys(USERNAME)
password_input.clear()
password_input.send_keys(PASSWORD)

# 5. Click the submit/login button
try:
    submit_btn = driver.find_element(By.XPATH, "//button[contains(.,'Log in') or contains(.,'Sign in') or @type='submit'] | //input[@type='submit']")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", submit_btn)
    submit_btn.click()
    print("🔐 Submitted login form")
except Exception as e:
    raise RuntimeError("⚠️ Couldn't click submit button") from e

# 6. Wait for confirmation of login success
try:
    wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(.,'My Stats Tables') or contains(.,'Player Stats')]")))
    print("✅ Login successful — continuing to player stats page...")
except Exception:
    raise RuntimeError("❌ Login failed — wrong credentials or captcha")




ℹ️ No cookies to accept
⚠️ Couldn't find login button: Message: 
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:555:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:136:16

✅ Login form detected
⚠️ Couldn't find login button: Message: 
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:555:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:136:16

✅ Login form detected
🔐 Submitted login form
✅ Login successful — continuing to player stats page...
🔐 Submitted login form
✅ Login successful — continuing to player stats page...


## 5. Navigate to Custom Stats Table

Navigate to the specific custom stats table URL after logging in.

In [92]:
# Περιμένουμε να φορτώσει το Members Area
wait.until(EC.presence_of_element_located((By.LINK_TEXT, "My Stats Tables")))
driver.get("https://members.fantasyfootballscout.co.uk/my-stats-tables/view/66933/")
print("✅ Accessed custom stats table.")

✅ Accessed custom stats table.


## 6. Scrape Data for Each Gameweek

Iterate through each gameweek, apply filters, scrape the HTML table using pandas, and append the data to a list.

In [93]:

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
import pandas as pd
import time

OUTPUT_PATH = Path("data/fpl_scout/2024-25/edw")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

driver.get("https://members.fantasyfootballscout.co.uk/my-stats-tables/view/66933/")
wait = WebDriverWait(driver, 20)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))
print("✅ My Stats Table loaded.")

all_data = []
print("📦 Initialized data collection list")

# Επιλογή "Gameweek Range" στο dropdown
try:
    range_dropdown = wait.until(EC.element_to_be_clickable((By.ID, "frange")))
    Select(range_dropdown).select_by_value("GAMEWEEK_RANGE")
    print("✅ Selected 'Gameweek Range' option")
    time.sleep(2)
except Exception as e:
    print(f"⚠️ Could not set range dropdown: {e}")

# Loop για κάθε Gameweek
for gw in range(1, 39):
    print(f"\n{'='*50}\n📊 GAMEWEEK {gw}\n{'='*50}")
    retry_count = 0
    max_retries = 3
    success = False

    while retry_count < max_retries and not success:
        try:
            # 🔑 Re-find elements to avoid stale references
            range_start = wait.until(EC.presence_of_element_located((By.ID, "fgameweek-start")))
            range_end = wait.until(EC.presence_of_element_located((By.ID, "fgameweek-end")))

            Select(range_start).select_by_value(str(gw))
            Select(range_end).select_by_value(str(gw))
            print(f"🔽 Selected GW {gw} in dropdowns")

            # Click Filter button
            filter_btn = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "input[type='submit'][name='filter'][value='Filter']"))
            )
            filter_btn.click()
            print("🔄 Clicked Filter button")

            # **Περιμένει να ανανεωθεί ο πίνακας** (table refresh check)
            wait.until(EC.staleness_of(driver.find_element(By.CSS_SELECTOR, "table tbody tr")))
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
            time.sleep(1)

            # Διαβάζουμε τον πίνακα
            tables = pd.read_html(driver.page_source)
            if not tables:
                print(f"⚠️ No tables found for GW {gw}")
                retry_count += 1
                continue

            df_gw = tables[0]
            # Προσθέτουμε τη στήλη Gameweek δίπλα από το Name
            if 'Name' in df_gw.columns:
                cols = df_gw.columns.tolist()
                idx = cols.index('Name') + 1
                df_gw.insert(idx, 'Gameweek', gw)
            else:
                df_gw['Gameweek'] = gw

            all_data.append(df_gw)
            print(f"✅ Saved {len(df_gw)} rows for GW {gw}")
            success = True

        except Exception as e:
            print(f"❌ Error GW {gw} (attempt {retry_count+1}/{max_retries}): {e}")
            retry_count += 1
            time.sleep(2)

    if not success:
        print(f"❌ Failed to scrape GW {gw} after {max_retries} attempts, skipping...")

✅ My Stats Table loaded.
📦 Initialized data collection list
✅ Selected 'Gameweek Range' option

📊 GAMEWEEK 1
🔽 Selected GW 1 in dropdowns

📊 GAMEWEEK 1
🔽 Selected GW 1 in dropdowns
🔄 Clicked Filter button
🔄 Clicked Filter button
❌ Error GW 1 (attempt 1/3): Message: 

❌ Error GW 1 (attempt 1/3): Message: 

🔽 Selected GW 1 in dropdowns
🔄 Clicked Filter button
🔽 Selected GW 1 in dropdowns
🔄 Clicked Filter button


C:\Users\SOFI\AppData\Local\Temp\ipykernel_5296\663605601.py:58: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(driver.page_source)


✅ Saved 307 rows for GW 1

📊 GAMEWEEK 2
🔽 Selected GW 2 in dropdowns
🔄 Clicked Filter button
🔄 Clicked Filter button
❌ Error GW 2 (attempt 1/3): Message: 

❌ Error GW 2 (attempt 1/3): Message: 

🔽 Selected GW 2 in dropdowns
🔄 Clicked Filter button
🔽 Selected GW 2 in dropdowns
🔄 Clicked Filter button


C:\Users\SOFI\AppData\Local\Temp\ipykernel_5296\663605601.py:58: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(driver.page_source)


✅ Saved 312 rows for GW 2

📊 GAMEWEEK 3
🔽 Selected GW 3 in dropdowns
🔄 Clicked Filter button
🔄 Clicked Filter button
❌ Error GW 3 (attempt 1/3): Message: 

❌ Error GW 3 (attempt 1/3): Message: 

🔽 Selected GW 3 in dropdowns
🔄 Clicked Filter button
🔽 Selected GW 3 in dropdowns
🔄 Clicked Filter button


...

C:\Users\SOFI\AppData\Local\Temp\ipykernel_5296\663605601.py:58: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(driver.page_source)


✅ Saved 305 rows for GW 36

📊 GAMEWEEK 37
🔽 Selected GW 37 in dropdowns
🔄 Clicked Filter button
🔄 Clicked Filter button
❌ Error GW 37 (attempt 1/3): Message: 

❌ Error GW 37 (attempt 1/3): Message: 

🔽 Selected GW 37 in dropdowns
🔄 Clicked Filter button
🔽 Selected GW 37 in dropdowns
🔄 Clicked Filter button


C:\Users\SOFI\AppData\Local\Temp\ipykernel_5296\663605601.py:58: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(driver.page_source)


✅ Saved 309 rows for GW 37

📊 GAMEWEEK 38
🔽 Selected GW 38 in dropdowns
🔄 Clicked Filter button
🔄 Clicked Filter button
❌ Error GW 38 (attempt 1/3): Message: 

❌ Error GW 38 (attempt 1/3): Message: 

🔽 Selected GW 38 in dropdowns
🔄 Clicked Filter button
🔽 Selected GW 38 in dropdowns
🔄 Clicked Filter button
✅ Saved 308 rows for GW 38
✅ Saved 308 rows for GW 38


C:\Users\SOFI\AppData\Local\Temp\ipykernel_5296\663605601.py:58: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(driver.page_source)


## 7. Save Data to CSV

Combine all gameweek data into a single DataFrame and save it as a CSV file in the specified output path.

In [94]:
# %% [markdown]
# ## 7. Save Data to CSV (updated)

if all_data:
    df = pd.concat(all_data, ignore_index=True)
    csv_path = OUTPUT_PATH / "mytable_scout_data.csv"
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n✅ DATA SAVED SUCCESSFULLY!")
    print(f"📁 File: {csv_path.name}")
    print(f"📂 Location: {csv_path.absolute()}")
    print(f"💾 Size: {csv_path.stat().st_size / 1024:.2f} KB")
    print("\n👀 PREVIEW:")
    print(df.head(10).to_string())
    print("\n📈 GAMEWEEK DISTRIBUTION:")
    print(df['Gameweek'].value_counts().sort_index())
else:
    print("\n❌ NO DATA COLLECTED")


✅ DATA SAVED SUCCESSFULLY!
📁 File: mytable_scout_data.csv
📂 Location: c:\Python\fpl_pipeline\data\fpl_scout\2024-25\edw\mytable_scout_data.csv
💾 Size: 551.12 KB

👀 PREVIEW:
   Unnamed: 0               Name  Gameweek Team  Cost  Corners  Goal Attempts  xA Expected Assists  xG Expected Goals  Big Assists  Chances Created  DC Pts  Ownership  Svs  Solo Goals
0         NaN         van de Ven         1  TOT   4.7        0              0                 0.02               0.00            0                0       0       32.6    0           0
1         NaN          van Hecke         1  BHA   4.4        0              0                 0.00               0.00            0                0       0        1.3    0           0
2         NaN           van Dijk         1  LIV   6.0        0              0                 0.01               0.00            0                0       0       23.6    0           0
3         NaN             Danilo         1  NFO   4.7        0              0             

## 8. Quit WebDriver

Close the Selenium WebDriver instance to release resources.

In [2]:
import pandas as pd
df = pd.read_csv("output/missing_OD_diagnostic.csv")
print(df.head(20))
print(df.columns)


         A                           Player UUID  Code  \
0    33675  5c29d002-f38e-4603-ba9a-01a4b8b8eed4   486   
1    33676  5c29d002-f38e-4603-ba9a-01a4b8b8eed4   486   
2    33677  5c29d002-f38e-4603-ba9a-01a4b8b8eed4   486   
3    33678  5c29d002-f38e-4603-ba9a-01a4b8b8eed4   486   
4   111929  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
5   111930  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
6   111931  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
7   111932  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
8   111939  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
9   111940  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
10  111941  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
11  111942  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
12  111943  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
13  111944  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
14  111945  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
15  111946  fdb3bfb8-1d33-4d22-86fa-e546c71b2187    70   
16  111947  fd

In [4]:
import pandas as pd
df = pd.read_csv("data/master_team_list.csv")
print(df.head(10))
print(df.columns)


    season  team       team_name
0  2016-17     1         Arsenal
1  2016-17     2     Bournemouth
2  2016-17     3         Burnley
3  2016-17     4         Chelsea
4  2016-17     5  Crystal Palace
5  2016-17     6         Everton
6  2016-17     7            Hull
7  2016-17     8       Leicester
8  2016-17     9       Liverpool
9  2016-17    10        Man City
Index(['season', 'team', 'team_name'], dtype='object')


In [5]:
import pandas as pd

m = pd.read_csv("data/master_team_list.csv")
print(m["season"].unique())


['2016-17' '2017-18' '2018-19' '2019-20' '2020-21' '2021-22' '2022-23'
 '2023-24']


In [95]:
driver.quit()